<a href="https://colab.research.google.com/github/Matthaios-Tsintsinis/thesis-rag-system/blob/main/hierarchical_rag_tsintsinis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hierarchical RAG pipeline for Colab

This notebook builds an end-to-end **Hierarchical Retrieval-Augmented Generation (RAG)** pipeline over a folder of documents from Google Drive.

## What it does
- mounts Google Drive and scans a folder of documents
- extracts text from PDF, DOCX, TXT, Markdown, HTML, CSV, JSON, XLSX
- cleans and chunks the text
- computes multilingual embeddings
- builds a flat FAISS index
- builds a **hierarchical clustering tree** over document chunks
- supports **hierarchical retrieval** with optional reranking
- generates grounded answers with citations
- exports a **JSON report** with corpus analytics, clustering diagnostics, retrieval diagnostics, and run metadata

## Defaults
- Embeddings: `BAAI/bge-m3`
- Reranker: `BAAI/bge-reranker-v2-m3`
- Local generator: `Qwen/Qwen2.5-1.5B-Instruct` (optional)
- Vector index: FAISS
- Clustering: agglomerative hierarchical clustering


## Notes

### Expected folder layout
Put all documents inside:

`/content/drive/MyDrive/tsintsinis`

Subfolders are supported.

### JSON outputs
The notebook writes results to:

`/content/drive/MyDrive/tsintsinis_outputs`

Typical files:
- `hierarchical_rag_report_<timestamp>.json`
- `single_query_result_<timestamp>.json`
- `evaluation_report_<timestamp>.json` (if you provide an evaluation file)

### Evaluation file format
If you want quantitative comparison between flat and hierarchical retrieval, create a JSON file such as:

```json
[
  {
    "question": "What is the main topic of document X?",
    "gold_answer": "Optional reference answer",
    "gold_doc_ids": ["doc_00012"]
  }
]
```



In [ ]:
# ============================================
# Clear uploads folder
# ============================================
RUN_CLEAR_UPLOADS = False  # set to True to run

if RUN_CLEAR_UPLOADS:
    import os
    import shutil

    upload_dir = "/content/uploads"

    if os.path.exists(upload_dir):
        files_deleted = 0
        for filename in os.listdir(upload_dir):
            filepath = os.path.join(upload_dir, filename)
            if os.path.isfile(filepath):
                os.remove(filepath)
                print(f"Deleted: {filename}")
                files_deleted += 1
            elif os.path.isdir(filepath):
                shutil.rmtree(filepath)
                print(f"Deleted folder: {filename}")
        print(f"\nDone — {files_deleted} file(s) deleted.")
    else:
        print("Uploads folder doesn't exist yet.")

In [ ]:
# ============================================
# Clear cache
# ============================================
RUN_CLEAR_CACHE = False

if RUN_CLEAR_CACHE:
    for f in ["embeddings.npy", "faiss.index", "chunks.json"]:
        path = f"/content/cache/{f}"
        if os.path.exists(path):
            os.remove(path)
            print(f"Deleted cache: {f}")

In [ ]:

# ============================================
# 1) Environment setup
# ============================================
!pip -q install -U pip
!pip -q install pymupdf python-docx pypdf beautifulsoup4 openpyxl pandas numpy scikit-learn scipy faiss-cpu sentence-transformers transformers accelerate bitsandbytes datasets rank-bm25 nltk tqdm unidecode

import os
import re
import gc
import io
import json
import math
import time
import uuid
import glob
import shutil
import random
import logging
import warnings
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    HAS_TORCH = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    HAS_TORCH = False
    DEVICE = "cpu"

print("DEVICE:", DEVICE)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.5 MB/s eta 0:00:00
DEVICE: cuda


In [ ]:

# ============================================
# 2) Configuration
# ============================================

#UPDATED
# ----------------------------------------------------------------------------

import os
from google.colab import files

# Upload your PDFs
print("Please upload your PDF files:")
uploaded = files.upload()

RUN_ID = time.strftime("%Y%m%d_%H%M%S")

# Save uploaded files to local folder
UPLOAD_DIR = "/content/uploads"
CACHE_DIR = "/content/cache"        # shared across all runs
OUTPUT_DIR = f"/content/outputs/{RUN_ID}" # each run gets its own subfolder
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

for filename, data in uploaded.items():
    with open(f"{UPLOAD_DIR}/{filename}", "wb") as f:
        f.write(data)
    print(f"Saved: {filename}")

# ----------------------------------------------------------------------------


CONFIG = {
    # Input / output
    "drive_input_folder": UPLOAD_DIR, #UPDATED
    "output_dir": OUTPUT_DIR, #UPDATED

    # Parsing / chunking
    "min_chars_per_doc": 200,
    "chunk_size_words": 220,
    "chunk_overlap_words": 40,
    "max_chunks_per_doc": None,
    "context_neighbor_radius": 1,

    # Models
    "embedding_model_name": "BAAI/bge-m3",
    "reranker_model_name": "BAAI/bge-reranker-v2-m3",
    "generator_model_name": "Qwen/Qwen2.5-3B-Instruct",

    # Retrieval
    "top_k_flat": 12,
    "top_k_final": 6,
    "top_docs_after_tree": 8,
    "top_chunks_per_doc_for_context": 3,
    "tree_branching_factor": 4,
    "tree_top_branches_per_level": 2,
    "tree_min_cluster_size": 24,
    "tree_max_depth": 4,
    "max_query_views": 5,
    "rerank_top_n": 24,
    "use_reranker": True,
    "use_bm25_hybrid": True,
    "alpha_dense": 0.75,

    # Query augmentation
    "enable_query_view_generation": True,

    # Generation
    "enable_local_generation": True,
    "max_new_tokens": 512,
    "temperature": 0.1,
    "do_sample": False,

    # Optional evaluation file
    "evaluation_json_path": None,

    # Query batch for reporting
    "analysis_queries": [
        "Ποια είναι τα βασικά θέματα που καλύπτουν τα έγγραφα;",
        "What are the main topics in the document collection?",
        "Give a summary of the most important entities, concepts, and recurring themes."
    ]
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))


Please upload your PDF files:


Saving 25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf to 25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf
Saving download.pdf to download.pdf
Saving Efficient and Robust Approximate Nearest .pdf to Efficient and Robust Approximate Nearest .pdf
Saving embeddings.pdf to embeddings.pdf
Saving ICLR-2024-self-rag-learning-to-retrieve-generate-and-critique-through-self-reflection-Paper-Conference.pdf to ICLR-2024-self-rag-learning-to-retrieve-generate-and-critique-through-self-reflection-Paper-Conference.pdf
Saving NeurIPS-2020-retrieval-augmented-generation-for-knowledge-intensive-nlp-tasks-Paper.pdf to NeurIPS-2020-retrieval-augmented-generation-for-knowledge-intensive-nlp-tasks-Paper.pdf
Saving Pascual - From Text to Vectors Building an Advanced Semantic Retrieval System with RAG Techniques.pdf to Pascual - From Text to Vectors Building an Advanced Semantic Retrieval System with RAG Techniques.pdf
Saving seq2seq.pdf to seq2seq.pdf
Saving survey-on-in-context-learning.pdf to survey-on-in-context-learnin

In [ ]:

# ============================================
# 3) Utilities and parsers
# ============================================
import fitz  # pymupdf
from bs4 import BeautifulSoup
from docx import Document as DocxDocument

SUPPORTED_EXTENSIONS = {
    ".pdf", ".txt", ".md", ".html", ".htm", ".docx",
    ".csv", ".json", ".xlsx"
}

def safe_read_text(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def parse_pdf(path: str) -> Tuple[str, Dict[str, Any]]:
    text_parts = []
    page_stats = []
    doc = fitz.open(path)
    meta = doc.metadata or {}
    for i, page in enumerate(doc):
        txt = page.get_text("text")
        text_parts.append(f"\n\n[PAGE {i+1}]\n{txt}")
        page_stats.append({"page": i+1, "chars": len(txt)})
    return "\n".join(text_parts), {"pdf_metadata": meta, "page_stats": page_stats, "n_pages": len(doc)}

def parse_docx(path: str) -> Tuple[str, Dict[str, Any]]:
    d = DocxDocument(path)
    paras = [p.text for p in d.paragraphs if p.text.strip()]
    return "\n".join(paras), {"n_paragraphs": len(paras)}

def parse_html(path: str) -> Tuple[str, Dict[str, Any]]:
    html = safe_read_text(path)
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.extract()
    text = soup.get_text("\n")
    return text, {}

def parse_csv(path: str) -> Tuple[str, Dict[str, Any]]:
    df = pd.read_csv(path)
    sample = df.head(200).astype(str)
    txt = "\n".join([" | ".join(row) for row in sample.values.tolist()])
    return txt, {"n_rows": int(df.shape[0]), "n_cols": int(df.shape[1]), "columns": list(map(str, df.columns))}

def parse_json_file(path: str) -> Tuple[str, Dict[str, Any]]:
    obj = json.load(open(path, "r", encoding="utf-8"))
    pretty = json.dumps(obj, ensure_ascii=False, indent=2)
    return pretty, {"root_type": type(obj).__name__}

def parse_xlsx(path: str) -> Tuple[str, Dict[str, Any]]:
    xls = pd.ExcelFile(path)
    parts = []
    sheet_shapes = {}
    for sheet in xls.sheet_names[:10]:
        df = pd.read_excel(path, sheet_name=sheet)
        sheet_shapes[sheet] = [int(df.shape[0]), int(df.shape[1])]
        sample = df.head(100).astype(str)
        parts.append(f"\n[SHEET: {sheet}]\n")
        parts.append("\n".join([" | ".join(row) for row in sample.values.tolist()]))
    return "\n".join(parts), {"sheet_shapes": sheet_shapes, "sheet_names": xls.sheet_names}

def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\u00a0", " ", text)
    return text.strip()

def extract_text(path: str) -> Tuple[str, Dict[str, Any]]:
    ext = os.path.splitext(path)[1].lower()
    metadata = {"path": path, "ext": ext}
    if ext == ".pdf":
        text, extra = parse_pdf(path)
    elif ext == ".docx":
        text, extra = parse_docx(path)
    elif ext in [".html", ".htm"]:
        text, extra = parse_html(path)
    elif ext == ".csv":
        text, extra = parse_csv(path)
    elif ext == ".json":
        text, extra = parse_json_file(path)
    elif ext == ".xlsx":
        text, extra = parse_xlsx(path)
    else:
        text, extra = safe_read_text(path), {}
    metadata.update(extra)
    return clean_text(text), metadata

def detect_page_refs(text: str, chunk_text: str) -> List[int]:
    matches = re.findall(r"\[PAGE (\d+)\]", chunk_text)
    return [int(m) for m in matches] if matches else []

def chunk_text_words(text: str, chunk_size_words: int = 220, overlap_words: int = 40) -> List[Dict[str, Any]]:
    words = text.split()
    chunks = []
    if not words:
        return chunks
    step = max(1, chunk_size_words - overlap_words)
    idx = 0
    chunk_id = 0
    while idx < len(words):
        sub = words[idx: idx + chunk_size_words]
        chunk_text = " ".join(sub).strip()
        if chunk_text:
            chunks.append({
                "chunk_id_local": chunk_id,
                "start_word": idx,
                "end_word": idx + len(sub),
                "text": chunk_text,
                "n_words": len(sub)
            })
            chunk_id += 1
        idx += step
    return chunks

def list_files_recursive(root: str) -> List[str]:
    files = []
    for path, _, fnames in os.walk(root):
        for f in fnames:
            full = os.path.join(path, f)
            ext = os.path.splitext(full)[1].lower()
            if ext in SUPPORTED_EXTENSIONS:
                files.append(full)
    return sorted(files)

def normalize_scores(arr):
    arr = np.asarray(arr, dtype=float)
    if len(arr) == 0:
        return arr
    mn, mx = arr.min(), arr.max()
    if mx - mn < 1e-9:
        return np.ones_like(arr)
    return (arr - mn) / (mx - mn)

def mkdir(path):
    os.makedirs(path, exist_ok=True)


In [ ]:

# ============================================
# 4) Ingestion and chunking
# ============================================
files = list_files_recursive(CONFIG["drive_input_folder"])
print(f"Found {len(files)} candidate files")

documents = []
chunks = []
skipped = []

for file_idx, path in enumerate(tqdm(files)):
    try:
        text, meta = extract_text(path)
        if len(text) < CONFIG["min_chars_per_doc"]:
            skipped.append({"path": path, "reason": f"text shorter than {CONFIG['min_chars_per_doc']} chars"})
            continue

        relpath = os.path.relpath(path, CONFIG["drive_input_folder"])
        ext = os.path.splitext(path)[1].lower()
        doc_id = f"doc_{file_idx:05d}"
        source_file_id = relpath.replace(os.sep, "__")
        doc_record = {
            "doc_id": doc_id,
            "source_file_id": source_file_id,
            "source_path": path,
            "relative_path": relpath,
            "file_name": os.path.basename(path),
            "ext": ext,
            "text": text,
            "n_chars": len(text),
            "n_words": len(text.split()),
            "metadata": meta
        }
        documents.append(doc_record)

        doc_chunks = chunk_text_words(
            text,
            chunk_size_words=CONFIG["chunk_size_words"],
            overlap_words=CONFIG["chunk_overlap_words"]
        )
        if CONFIG["max_chunks_per_doc"] is not None:
            doc_chunks = doc_chunks[:CONFIG["max_chunks_per_doc"]]

        for ch in doc_chunks:
            global_chunk_id = f"{doc_id}_ch_{ch['chunk_id_local']:04d}"
            page_refs = detect_page_refs(text, ch["text"])
            chunks.append({
                "chunk_id": global_chunk_id,
                "chunk_id_local": int(ch["chunk_id_local"]),
                "doc_id": doc_id,
                "source_file_id": source_file_id,
                "source_path": path,
                "relative_path": relpath,
                "file_name": os.path.basename(path),
                "ext": ext,
                "text": ch["text"],
                "n_words": int(ch["n_words"]),
                "start_word": int(ch["start_word"]),
                "end_word": int(ch["end_word"]),
                "page_refs": page_refs
            })
    except Exception as e:
        skipped.append({"path": path, "reason": repr(e)})

documents_df = pd.DataFrame([{k: v for k, v in d.items() if k != "text"} for d in documents])
chunks_df = pd.DataFrame([{k: v for k, v in c.items() if k != "text"} for c in chunks])

doc_chunks_map = defaultdict(list)
for c in chunks:
    doc_chunks_map[c["doc_id"]].append(c)
for doc_id in doc_chunks_map:
    doc_chunks_map[doc_id] = sorted(doc_chunks_map[doc_id], key=lambda x: x["chunk_id_local"])

print("Documents:", len(documents))
print("Chunks:", len(chunks))
print("Skipped:", len(skipped))
display(documents_df.head(3))
display(chunks_df.head(3))


Found 12 candidate files


  0%|          | 0/12 [00:00<?, ?it/s]

Documents: 12
Chunks: 740
Skipped: 0


,doc_id,source_file_id,source_path,relative_path,file_name,ext,n_chars,n_words,metadata
0,doc_00000,25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf,/content/uploads/25a43194-c74c-4cd3-b60f-0a1f2...,25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf,25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf,.pdf,136566,19872,{'path': '/content/uploads/25a43194-c74c-4cd3-...
1,doc_00001,Efficient and Robust Approximate Nearest .pdf,/content/uploads/Efficient and Robust Approxim...,Efficient and Robust Approximate Nearest .pdf,Efficient and Robust Approximate Nearest .pdf,.pdf,67586,10797,{'path': '/content/uploads/Efficient and Robus...
2,doc_00002,ICLR-2024-self-rag-learning-to-retrieve-genera...,/content/uploads/ICLR-2024-self-rag-learning-t...,ICLR-2024-self-rag-learning-to-retrieve-genera...,ICLR-2024-self-rag-learning-to-retrieve-genera...,.pdf,107541,16528,{'path': '/content/uploads/ICLR-2024-self-rag-...


,chunk_id,chunk_id_local,doc_id,source_file_id,source_path,relative_path,file_name,ext,n_words,start_word,end_word,page_refs
0,doc_00000_ch_0000,0,doc_00000,25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf,/content/uploads/25a43194-c74c-4cd3-b60f-0a1f2...,25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf,25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf,.pdf,220,0,220,[1]
1,doc_00000_ch_0001,1,doc_00000,25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf,/content/uploads/25a43194-c74c-4cd3-b60f-0a1f2...,25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf,25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf,.pdf,220,180,400,[]
2,doc_00000_ch_0002,2,doc_00000,25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf,/content/uploads/25a43194-c74c-4cd3-b60f-0a1f2...,25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf,25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf,.pdf,220,360,580,[]


In [ ]:

# ============================================
# 5) Load models
# ============================================
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

print("Loading embedding model:", CONFIG["embedding_model_name"])
embedder = SentenceTransformer(CONFIG["embedding_model_name"], device=DEVICE)

reranker = None
if CONFIG["use_reranker"]:
    print("Loading reranker:", CONFIG["reranker_model_name"])
    reranker = CrossEncoder(CONFIG["reranker_model_name"], device=DEVICE)

tokenized_corpus = [c["text"].lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_corpus) if CONFIG["use_bm25_hybrid"] else None

generator = None
tokenizer = None
if CONFIG["enable_local_generation"]:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    print("Loading generator:", CONFIG["generator_model_name"])
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["generator_model_name"])
    kwargs = {"device_map": "auto"} if DEVICE == "cuda" else {}
    if DEVICE == "cuda":
        try:
            from transformers import BitsAndBytesConfig
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True, # nested quant → slightly smaller
                bnb_4bit_quant_type="nf4",  # best for LLMs
            )
            print("4-bit quantization enabled (NF4)")
        except Exception as e:
            kwargs["torch_dtype"] = torch.float16
            print(f"bitsandbytes not available, falling back to fp16: {e}")
    generator = AutoModelForCausalLM.from_pretrained(CONFIG["generator_model_name"], **kwargs)

print("Models ready")


Loading embedding model: BAAI/bge-m3


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Models ready


In [ ]:
#UPDATED (cache embeddings)
# ============================================
# 6) Embeddings and FAISS index
# ============================================
import faiss
import os

CACHE_EMBEDDINGS = os.path.join(CACHE_DIR, "embeddings.npy")
CACHE_INDEX = os.path.join(CACHE_DIR, "faiss.index")
CACHE_CHUNKS = os.path.join(CACHE_DIR, "chunks.json")

chunk_texts = [c["text"] for c in chunks]

if os.path.exists(CACHE_EMBEDDINGS) and os.path.exists(CACHE_INDEX) and os.path.exists(CACHE_CHUNKS):
    print("Loading cached embeddings and index...")
    chunk_embeddings = np.load(CACHE_EMBEDDINGS)
    index = faiss.read_index(CACHE_INDEX)
    print("Loaded from cache.")
else:
    print("No cache found — computing embeddings from scratch...")
    chunk_embeddings = embedder.encode(
        chunk_texts,
        batch_size=16,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    dim = chunk_embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(chunk_embeddings)

    # save to cache
    np.save(CACHE_EMBEDDINGS, chunk_embeddings)
    faiss.write_index(index, CACHE_INDEX)
    print("Saved embeddings and index to cache.")

chunk_id_to_idx = {c["chunk_id"]: i for i, c in enumerate(chunks)}
doc_id_to_idx = {d["doc_id"]: i for i, d in enumerate(documents)}

print("Embeddings shape:", chunk_embeddings.shape)

No cache found — computing embeddings from scratch...


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Saved embeddings and index to cache.
Embeddings shape: (740, 1024)


In [ ]:

# ============================================
# 7) Semantic tree construction (hierarchical clustering)
# ============================================
from sklearn.cluster import MiniBatchKMeans
from sklearn.feature_extraction.text import TfidfVectorizer

def compute_centroid(vectors):
    centroid = np.mean(vectors, axis=0)
    norm = np.linalg.norm(centroid)
    if norm > 0:
        centroid = centroid / norm
    return centroid.astype("float32")

def safe_cluster_keywords(texts, top_n=10):
    texts = [t for t in texts if isinstance(t, str) and t.strip()]
    if not texts:
        return []
    try:
        tfidf = TfidfVectorizer(max_features=5000, stop_words=None)
        X = tfidf.fit_transform(texts)
        sums = np.asarray(X.sum(axis=0)).ravel()
        feats = np.array(tfidf.get_feature_names_out())
        order = np.argsort(-sums)[:top_n]
        return feats[order].tolist()
    except Exception:
        return []

semantic_tree_nodes = {}
tree_leaf_to_chunk_indices = defaultdict(list)
tree_root_id = None

def build_semantic_tree(member_indices, depth=0, parent_id=None, path_tokens=None):
    global tree_root_id
    if path_tokens is None:
        path_tokens = []

    node_id = "root" if parent_id is None else "__".join(path_tokens)
    node_vectors = chunk_embeddings[member_indices]
    node_texts = [chunks[i]["text"] for i in member_indices]
    node_centroid = compute_centroid(node_vectors)
    node_keywords = safe_cluster_keywords(node_texts, top_n=8)

    node = {
        "node_id": node_id,
        "parent_id": parent_id,
        "depth": depth,
        "size": int(len(member_indices)),
        "member_indices": list(map(int, member_indices)),
        "centroid": node_centroid,
        "keywords": node_keywords,
        "children": []
    }
    semantic_tree_nodes[node_id] = node
    if parent_id is None:
        tree_root_id = node_id

    should_stop = (
        depth >= CONFIG["tree_max_depth"] or
        len(member_indices) < CONFIG["tree_min_cluster_size"] or
        len(member_indices) <= CONFIG["tree_branching_factor"]
    )
    if should_stop:
        tree_leaf_to_chunk_indices[node_id].extend(list(map(int, member_indices)))
        return node_id

    n_clusters = min(CONFIG["tree_branching_factor"], max(2, len(member_indices) // CONFIG["tree_min_cluster_size"]))
    local_vectors = chunk_embeddings[member_indices]
    km = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=min(2048, len(member_indices)))
    local_labels = km.fit_predict(local_vectors)

    unique_labels = sorted(set(local_labels.tolist()))
    # guard against collapsed clusters
    if len(unique_labels) <= 1:
        tree_leaf_to_chunk_indices[node_id].extend(list(map(int, member_indices)))
        return node_id

    for child_rank, label in enumerate(unique_labels):
        child_indices = [member_indices[i] for i, lab in enumerate(local_labels) if int(lab) == int(label)]
        if not child_indices:
            continue
        child_path = path_tokens + [f"d{depth+1}n{child_rank}"]
        child_id = build_semantic_tree(
            member_indices=child_indices,
            depth=depth + 1,
            parent_id=node_id,
            path_tokens=child_path
        )
        node["children"].append(child_id)

    if not node["children"]:
        tree_leaf_to_chunk_indices[node_id].extend(list(map(int, member_indices)))
    return node_id

root_member_indices = list(range(len(chunks)))
build_semantic_tree(root_member_indices, depth=0, parent_id=None, path_tokens=[])

chunk_idx_to_leaf_nodes = defaultdict(list)
for leaf_id, member_indices in tree_leaf_to_chunk_indices.items():
    for idx in member_indices:
        chunk_idx_to_leaf_nodes[int(idx)].append(leaf_id)

for idx, c in enumerate(chunks):
    leafs = chunk_idx_to_leaf_nodes.get(idx, [])
    c["tree_leaf_nodes"] = leafs
    c["primary_tree_leaf"] = leafs[0] if leafs else "root"

tree_summary_rows = []
for node_id, node in semantic_tree_nodes.items():
    tree_summary_rows.append({
        "node_id": node_id,
        "parent_id": node["parent_id"],
        "depth": node["depth"],
        "size": node["size"],
        "keywords": ", ".join(node["keywords"][:6]),
        "n_children": len(node["children"])
    })
tree_summary_df = pd.DataFrame(tree_summary_rows).sort_values(["depth", "size"], ascending=[True, False]).reset_index(drop=True)

print(f"Semantic tree nodes: {len(semantic_tree_nodes)}")
print(f"Leaf nodes: {len(tree_leaf_to_chunk_indices)}")
display(tree_summary_df.head(20))


Semantic tree nodes: 74
Leaf nodes: 44


,node_id,parent_id,depth,size,keywords,n_children
0,root,None,0,740,"the, of, and, to, in, is",4
1,d1n3,root,1,291,"the, and, to, of, in, rag",4
2,d1n1,root,1,207,"the, of, and, to, in, is",4
3,d1n2,root,1,160,"and, in, et, al, the, arxiv",4
4,d1n0,root,1,82,"the, of, is, to, layer, output",3
5,d1n3__d2n1,d1n3,2,116,"the, and, of, to, in, rag",4
6,d1n3__d2n0,d1n3,2,89,"the, and, to, we, of, rag",3
7,d1n2__d2n3,d1n2,2,76,"arxiv, and, in, al, et, of",3
8,d1n1__d2n1,d1n1,2,71,"the, and, of, to, in, vector",2
9,d1n3__d2n3,d1n3,2,56,"the, and, of, to, in, for",2


In [ ]:

# ============================================
# 8) Retrieval functions
# ============================================
def cosine_sim(a, b):
    a = np.asarray(a, dtype="float32")
    b = np.asarray(b, dtype="float32")
    return float(np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + 1e-9))

def generate_query_views(query: str, max_views: int = None) -> List[str]:
    max_views = max_views or CONFIG["max_query_views"]
    q = query.strip()
    views = [q]

    # deterministic semantic perspectives
    templates = [
        "main topic of: {q}",
        "key entities and concepts in: {q}",
        "evidence and passages relevant to: {q}",
        "sections discussing: {q}",
    ]
    for tpl in templates:
        if len(views) >= max_views:
            break
        views.append(tpl.format(q=q))

    # optional LLM-generated paraphrases for richer semantic coverage
    if CONFIG["enable_query_view_generation"] and generator is not None and tokenizer is not None and len(views) < max_views:
        prompt = f"""Create up to {max_views - len(views)} short retrieval-oriented reformulations of the question below.
Each reformulation should target a different semantic angle (topic, entities, evidence, terminology).
Return one per line and no numbering.

Question: {q}
"""
        try:
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
            if DEVICE == "cuda":
                inputs = {k: v.to(generator.device) for k, v in inputs.items()}
            with torch.no_grad():
                out = generator.generate(
                    **inputs,
                    max_new_tokens=128,
                    temperature=0.2,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )
            raw = tokenizer.decode(out[0], skip_special_tokens=True)
            tail = raw.split("Question:", 1)[-1]
            for line in tail.splitlines():
                cand = line.strip(" -*\t")
                if cand and cand.lower() != q.lower() and len(cand.split()) >= 3:
                    views.append(cand)
                    if len(views) >= max_views:
                        break
        except Exception as e:
            print("Query view generation fallback:", repr(e))

    dedup = []
    seen = set()
    for v in views:
        key = v.strip().lower()
        if key and key not in seen:
            seen.add(key)
            dedup.append(v.strip())
        if len(dedup) >= max_views:
            break
    return dedup

def dense_search_single_query(query: str, top_k: int = 10):
    q_emb = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")[0]
    scores, idxs = index.search(np.asarray([q_emb]), top_k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        c = chunks[int(idx)]
        results.append({
            "idx": int(idx),
            "chunk_id": c["chunk_id"],
            "doc_id": c["doc_id"],
            "source_file_id": c["source_file_id"],
            "relative_path": c["relative_path"],
            "text": c["text"],
            "score_dense": float(score),
            "primary_tree_leaf": c["primary_tree_leaf"],
            "page_refs": c["page_refs"],
        })
    return results, q_emb

def aggregate_multiview_dense_search(query_views: List[str], top_k_per_view: int = 10):
    all_hits = {}
    query_embeddings = {}
    for qv in query_views:
        view_hits, q_emb = dense_search_single_query(qv, top_k=top_k_per_view)
        query_embeddings[qv] = q_emb
        for rank, hit in enumerate(view_hits, start=1):
            key = hit["idx"]
            rec = all_hits.setdefault(key, {
                **hit,
                "score_dense_max": -1.0,
                "score_dense_mean": 0.0,
                "view_hits": []
            })
            rec["view_hits"].append({"query_view": qv, "rank": rank, "score_dense": hit["score_dense"]})
            rec["score_dense_max"] = max(rec["score_dense_max"], hit["score_dense"])
    for rec in all_hits.values():
        vals = [x["score_dense"] for x in rec["view_hits"]]
        rec["score_dense_mean"] = float(np.mean(vals))
    return sorted(all_hits.values(), key=lambda x: x["score_dense_max"], reverse=True), query_embeddings

def traverse_semantic_tree(query_emb: np.ndarray):
    debug_steps = []
    active_nodes = [tree_root_id]
    collected_leaf_nodes = set()

    while active_nodes:
        next_nodes = []
        for node_id in active_nodes:
            node = semantic_tree_nodes[node_id]
            child_ids = node["children"]
            if not child_ids:
                collected_leaf_nodes.add(node_id)
                debug_steps.append({
                    "node_id": node_id,
                    "depth": node["depth"],
                    "decision": "leaf",
                    "size": node["size"],
                    "keywords": node["keywords"][:6]
                })
                continue

            child_scores = []
            for child_id in child_ids:
                child = semantic_tree_nodes[child_id]
                score = cosine_sim(query_emb, child["centroid"])
                child_scores.append((child_id, score, child["keywords"][:6], child["size"]))
            child_scores = sorted(child_scores, key=lambda x: x[1], reverse=True)
            chosen = child_scores[:CONFIG["tree_top_branches_per_level"]]
            debug_steps.append({
                "node_id": node_id,
                "depth": node["depth"],
                "decision": "expand",
                "candidates": [
                    {"child_id": cid, "score": float(sc), "keywords": kws, "size": int(sz)}
                    for cid, sc, kws, sz in child_scores
                ],
                "selected_children": [cid for cid, _, _, _ in chosen]
            })
            next_nodes.extend([cid for cid, _, _, _ in chosen])
        active_nodes = next_nodes

    candidate_indices = sorted(set(idx for leaf_id in collected_leaf_nodes for idx in tree_leaf_to_chunk_indices[leaf_id]))
    return candidate_indices, {
        "selected_leaf_nodes": sorted(collected_leaf_nodes),
        "steps": debug_steps
    }

def score_candidates_for_query_views(candidate_indices: List[int], query_views: List[str], query_embeddings: Dict[str, np.ndarray]):
    if not candidate_indices:
        return []

    bm25_scores_by_view = {}
    if bm25 is not None:
        for qv in query_views:
            bm25_scores_by_view[qv] = bm25.get_scores(qv.lower().split())

    scored = []
    for idx in candidate_indices:
        c = chunks[int(idx)]
        dense_vals = [float(np.dot(query_embeddings[qv], chunk_embeddings[idx])) for qv in query_views]
        dense_best = max(dense_vals) if dense_vals else 0.0
        dense_mean = float(np.mean(dense_vals)) if dense_vals else 0.0

        bm25_vals = []
        if bm25 is not None:
            for qv in query_views:
                bm25_vals.append(float(bm25_scores_by_view[qv][idx]))
        bm25_best = max(bm25_vals) if bm25_vals else 0.0

        scored.append({
            "idx": int(idx),
            "chunk_id": c["chunk_id"],
            "doc_id": c["doc_id"],
            "source_file_id": c["source_file_id"],
            "relative_path": c["relative_path"],
            "text": c["text"],
            "page_refs": c["page_refs"],
            "primary_tree_leaf": c["primary_tree_leaf"],
            "score_dense_best": dense_best,
            "score_dense_mean": dense_mean,
            "score_bm25_best": bm25_best,
        })

    dense_norm = normalize_scores([r["score_dense_best"] for r in scored])
    bm25_norm = normalize_scores([r["score_bm25_best"] for r in scored]) if bm25 is not None else np.zeros(len(scored))
    for i, r in enumerate(scored):
        r["score_hybrid"] = float(CONFIG["alpha_dense"] * dense_norm[i] + (1 - CONFIG["alpha_dense"]) * bm25_norm[i])
    return sorted(scored, key=lambda x: x["score_hybrid"], reverse=True)

def collect_source_documents(scored_candidates: List[Dict[str, Any]], top_docs: int = None):
    top_docs = top_docs or CONFIG["top_docs_after_tree"]
    doc_scores = {}
    for hit in scored_candidates:
        rec = doc_scores.setdefault(hit["doc_id"], {
            "doc_id": hit["doc_id"],
            "source_file_id": hit["source_file_id"],
            "relative_path": hit["relative_path"],
            "score": -1.0,
            "supporting_chunks": []
        })
        rec["score"] = max(rec["score"], hit["score_hybrid"])
        rec["supporting_chunks"].append({
            "chunk_id": hit["chunk_id"],
            "score_hybrid": hit["score_hybrid"],
            "primary_tree_leaf": hit["primary_tree_leaf"],
            "page_refs": hit["page_refs"]
        })
    ranked_docs = sorted(doc_scores.values(), key=lambda x: x["score"], reverse=True)
    for d in ranked_docs:
        d["supporting_chunks"] = sorted(d["supporting_chunks"], key=lambda x: x["score_hybrid"], reverse=True)
    return ranked_docs[:top_docs]

def expand_context_within_documents(selected_docs: List[Dict[str, Any]], scored_candidates: List[Dict[str, Any]]):
    candidate_by_chunk_id = {x["chunk_id"]: x for x in scored_candidates}
    context_records = []
    seen = set()

    for doc in selected_docs:
        doc_id = doc["doc_id"]
        doc_chunks = doc_chunks_map.get(doc_id, [])
        top_support = doc["supporting_chunks"][:CONFIG["top_chunks_per_doc_for_context"]]
        for support in top_support:
            support_chunk = next((c for c in doc_chunks if c["chunk_id"] == support["chunk_id"]), None)
            if support_chunk is None:
                continue
            center = support_chunk["chunk_id_local"]
            left = max(0, center - CONFIG["context_neighbor_radius"])
            right = min(len(doc_chunks) - 1, center + CONFIG["context_neighbor_radius"])
            window_chunks = doc_chunks[left:right+1]
            fragment_text = "\n".join([wc["text"] for wc in window_chunks]).strip()
            page_refs = sorted(set(p for wc in window_chunks for p in wc.get("page_refs", [])))
            fragment_id = f"{doc_id}::window::{left}-{right}"
            if fragment_id in seen:
                continue
            seen.add(fragment_id)
            context_records.append({
                "fragment_id": fragment_id,
                "doc_id": doc_id,
                "source_file_id": doc["source_file_id"],
                "relative_path": doc["relative_path"],
                "anchor_chunk_id": support["chunk_id"],
                "window_chunk_ids": [wc["chunk_id"] for wc in window_chunks],
                "page_refs": page_refs,
                "score_hybrid": float(candidate_by_chunk_id.get(support["chunk_id"], {}).get("score_hybrid", support["score_hybrid"])),
                "text": fragment_text
            })

    return sorted(context_records, key=lambda x: x["score_hybrid"], reverse=True)

def rerank_results(query: str, results: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    if reranker is None or not results:
        return results
    pairs = [(query, r["text"]) for r in results[:CONFIG["rerank_top_n"]]]
    rr_scores = reranker.predict(pairs, batch_size=16, show_progress_bar=False)
    for r, s in zip(results[:CONFIG["rerank_top_n"]], rr_scores):
        r["score_rerank"] = float(s)
    head = sorted(results[:CONFIG["rerank_top_n"]], key=lambda x: x.get("score_rerank", -1e9), reverse=True)
    tail = results[CONFIG["rerank_top_n"]:]
    return head + tail

def retrieve(query: str, mode: str = "hierarchical", top_k: int = None):
    top_k = top_k or CONFIG["top_k_final"]

    if mode == "flat":
        query_views = generate_query_views(query)
        mv_hits, query_embeddings = aggregate_multiview_dense_search(query_views, top_k_per_view=max(top_k, CONFIG["top_k_flat"]))
        top_hits = mv_hits[:max(top_k, CONFIG["top_k_flat"])]
        if bm25 is not None and top_hits:
            bm25_scores = bm25.get_scores(query.lower().split())
            bm25_norm = normalize_scores([bm25_scores[h["idx"]] for h in top_hits])
            dense_norm = normalize_scores([h["score_dense_max"] for h in top_hits])
            for i, h in enumerate(top_hits):
                h["score_hybrid"] = float(CONFIG["alpha_dense"] * dense_norm[i] + (1 - CONFIG["alpha_dense"]) * bm25_norm[i])
        else:
            for h in top_hits:
                h["score_hybrid"] = float(h["score_dense_max"])
        top_hits = sorted(top_hits, key=lambda x: x["score_hybrid"], reverse=True)
        if reranker is not None:
            top_hits = rerank_results(query, top_hits)
        return top_hits[:top_k], {
            "query_views": query_views,
            "mode": mode
        }

    query_views = generate_query_views(query)
    _, query_embeddings = aggregate_multiview_dense_search(query_views, top_k_per_view=4)
    base_query_emb = np.mean(np.vstack([query_embeddings[qv] for qv in query_views]), axis=0)
    base_query_emb = base_query_emb / (np.linalg.norm(base_query_emb) + 1e-9)

    candidate_indices, tree_debug = traverse_semantic_tree(base_query_emb)
    scored_candidates = score_candidates_for_query_views(candidate_indices, query_views, query_embeddings)
    selected_docs = collect_source_documents(scored_candidates, top_docs=CONFIG["top_docs_after_tree"])
    context_fragments = expand_context_within_documents(selected_docs, scored_candidates)

    if reranker is not None:
        context_fragments = rerank_results(query, context_fragments)

    final_results = context_fragments[:top_k]
    debug = {
        "query_views": query_views,
        "tree_traversal": tree_debug,
        "n_tree_candidates": len(candidate_indices),
        "selected_documents": selected_docs,
        "candidate_preview": scored_candidates[:20]
    }
    return final_results, debug


In [ ]:

# ============================================
# 9) Grounded answer generation with citations
# ============================================
def format_context(results: List[Dict[str, Any]]) -> str:
    blocks = []
    for i, r in enumerate(results, start=1):
        page_part = f" | pages={r['page_refs']}" if r.get("page_refs") else ""
        blocks.append(
            f"[SOURCE {i}] file={r['relative_path']} | doc_id={r['doc_id']} | anchor_chunk_id={r.get('anchor_chunk_id', '')}{page_part}\n{r['text']}"
        )
    return "\n\n".join(blocks)

def generate_grounded_answer(query: str, results: List[Dict[str, Any]]) -> Dict[str, Any]:
    context = format_context(results)

    prompt = f"""You are a careful research assistant.
Answer the question ONLY from the provided sources.
If the evidence is insufficient, say so explicitly.
Every factual claim should be grounded in the sources and cite one or more source numbers like [SOURCE 1].
Prefer concise synthesis over copying.
Return the answer in the same language as the user's question when possible.

Question:
{query}

Sources:
{context}

Answer:
"""

    if generator is None or tokenizer is None:
        answer_text = "Generation disabled. Retrieved evidence is available in the JSON output."
    else:
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096)
        if DEVICE == "cuda":
            inputs = {k: v.to(generator.device) for k, v in inputs.items()}
        with torch.no_grad():
            out = generator.generate(
                **inputs,
                max_new_tokens=CONFIG["max_new_tokens"],
                temperature=CONFIG["temperature"],
                do_sample=CONFIG["do_sample"],
                pad_token_id=tokenizer.eos_token_id
            )
        answer_text = tokenizer.decode(out[0], skip_special_tokens=True)
        if "Answer:" in answer_text:
            answer_text = answer_text.split("Answer:", 1)[-1].strip()

    citations = []
    for i, r in enumerate(results, start=1):
        citations.append({
            "source_no": i,
            "doc_id": r["doc_id"],
            "source_file_id": r["source_file_id"],
            "relative_path": r["relative_path"],
            "anchor_chunk_id": r.get("anchor_chunk_id"),
            "window_chunk_ids": r.get("window_chunk_ids", []),
            "page_refs": r.get("page_refs", []),
        })
    return {"answer": answer_text, "citations": citations}

#UPDATED (confidence)

CONFIDENCE_THRESHOLD = 0.35  # lowest confidence allowed before "I don't know" answer

def ask(query: str, mode: str = "hierarchical", top_k: int = None) -> Dict[str, Any]:
    results, debug = retrieve(query, mode=mode, top_k=top_k or CONFIG["top_k_final"])

    # check confidence of top result
    top_score = None
    if results:
        top_score = results[0].get("score_rerank") or results[0].get("score_hybrid") or None

    low_confidence = (top_score is None) or (top_score < CONFIDENCE_THRESHOLD)

    if low_confidence:
        print(f"⚠️  Low confidence (score={top_score}). No relevant content found for this query.")
        return {
            "run_id": RUN_ID,
            "query": query,
            "mode": mode,
            "answer": "I could not find relevant information in the provided documents to answer this question.",
            "confidence": top_score,
            "low_confidence": True,
            "citations": [],
            "retrieved_fragments": results,
            "debug": debug
        }

    gen = generate_grounded_answer(query, results)
    payload = {
        "run_id": RUN_ID,
        "query": query,
        "mode": mode,
        "answer": gen["answer"],
        "confidence": top_score,
        "low_confidence": False,
        "citations": gen["citations"],
        "retrieved_fragments": results,
        "selected_documents": debug.get("selected_documents", []),  #UPDATED
        "debug": debug
    }
    return payload


In [ ]:

# ============================================
# 10) Corpus analytics and JSON report
# ============================================
def corpus_report() -> Dict[str, Any]:
    ext_counts = Counter([os.path.splitext(d["relative_path"])[1].lower() for d in documents])
    words_per_doc = [d["n_words"] for d in documents]
    chars_per_doc = [d["n_chars"] for d in documents]
    chunks_per_doc = Counter([c["doc_id"] for c in chunks])
    leaf_sizes = Counter([c["primary_tree_leaf"] for c in chunks])

    return {
        "run_id": RUN_ID,
        "config": CONFIG,
        "device": DEVICE,
        "counts": {
            "n_input_files_detected": len(files),
            "n_documents_ingested": len(documents),
            "n_chunks": len(chunks),
            "n_skipped": len(skipped),
            "n_tree_nodes": len(semantic_tree_nodes),
            "n_tree_leaves": len(tree_leaf_to_chunk_indices),
        },
        "document_statistics": {
            "file_extensions": dict(ext_counts),
            "words_per_doc": {
                "min": int(np.min(words_per_doc)) if words_per_doc else 0,
                "max": int(np.max(words_per_doc)) if words_per_doc else 0,
                "mean": float(np.mean(words_per_doc)) if words_per_doc else 0.0,
                "median": float(np.median(words_per_doc)) if words_per_doc else 0.0,
            },
            "chars_per_doc": {
                "min": int(np.min(chars_per_doc)) if chars_per_doc else 0,
                "max": int(np.max(chars_per_doc)) if chars_per_doc else 0,
                "mean": float(np.mean(chars_per_doc)) if chars_per_doc else 0.0,
                "median": float(np.median(chars_per_doc)) if chars_per_doc else 0.0,
            },
            "chunks_per_doc": {
                "min": int(min(chunks_per_doc.values())) if chunks_per_doc else 0,
                "max": int(max(chunks_per_doc.values())) if chunks_per_doc else 0,
                "mean": float(np.mean(list(chunks_per_doc.values()))) if chunks_per_doc else 0.0,
                "median": float(np.median(list(chunks_per_doc.values()))) if chunks_per_doc else 0.0,
            },
        },
        "semantic_tree": {
            "root_id": tree_root_id,
            "largest_leaves": dict(leaf_sizes.most_common(20)),
            "sample_nodes": tree_summary_rows[:30],
        },
        "skipped": skipped[:200],
        "sample_documents": [
            {k: v for k, v in d.items() if k != "text"} for d in documents[:20]
        ],
        "analysis_queries": []
    }

report = corpus_report()
for q in CONFIG["analysis_queries"]:
    report["analysis_queries"].append(ask(q, mode="hierarchical", top_k=CONFIG["top_k_final"]))

report_path = os.path.join(CONFIG["output_dir"], f"corpus_report_{RUN_ID}.json")
json.dump(report, open(report_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Saved corpus report to:", report_path)


⚠️  Low confidence (score=0.025790423154830933). No relevant content found for this query.
⚠️  Low confidence (score=0.16974204778671265). No relevant content found for this query.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Saved corpus report to: /content/outputs/20260427_181401/corpus_report_20260427_181401.json


In [ ]:
# ============================================
# 11) RAGAS evaluation - setup
# ============================================
# RAGAS scores RAG quality with LLM-as-judge metrics:
#   * faithfulness          - is the answer grounded in the retrieved contexts?
#   * response_relevancy    - does the answer actually address the question?
#   * context_precision     - are the retrieved contexts on-topic?
#   * context_recall        - did we retrieve enough?  (needs gold_answer)
#   * factual_correctness   - factual + semantic match (needs gold_answer)
#
# A *separate* judge LLM is used for evaluation. The local Qwen-3B
# is not a reliable LLM-as-judge — using a stronger model here keeps
# the *evaluation* trustworthy while generation stays local.
# Judge: OpenAI gpt-4o-mini (cheap, multilingual — handles Greek well).

!pip -q install "ragas>=0.2,<0.4" "langchain>=0.3" "langchain-openai>=0.2" openai rapidfuzz

import os
from getpass import getpass

# Judge LLM: OpenAI gpt-4o-mini — cheap (~$0.10–0.30 per full eval run),
# Generation still happens with the local Qwen 7B.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper

judge_chat = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_llm = LangchainLLMWrapper(judge_chat)
print("Judge LLM: OpenAI gpt-4o-mini")

# ----- Reuse the BAAI/bge-m3 embedder already loaded as `embedder` -----
# Some metrics (response_relevancy) need embeddings. We wrap the in-memory
# SentenceTransformer with the LangChain Embeddings interface so RAGAS can
# use it without loading a second copy of bge-m3 (keeps multilingual / Greek
# behaviour identical to retrieval).
from langchain_core.embeddings import Embeddings
from ragas.embeddings import LangchainEmbeddingsWrapper

class _ReuseSTEmbeddings(Embeddings):
    """LangChain Embeddings backed by the already-loaded SentenceTransformer."""
    def __init__(self, st_model):
        self.st = st_model
    def embed_documents(self, texts):
        return self.st.encode(
            list(texts), normalize_embeddings=True,
            batch_size=32, show_progress_bar=False
        ).tolist()
    def embed_query(self, text):
        return self.st.encode([text], normalize_embeddings=True)[0].tolist()

judge_embeddings = LangchainEmbeddingsWrapper(_ReuseSTEmbeddings(embedder))

print(f"RAGAS judge ready (gpt-4o-mini); embeddings reuse {CONFIG['embedding_model_name']}.")


Judge LLM: OpenAI gpt-4o-mini
RAGAS judge ready (gpt-4o-mini); embeddings reuse BAAI/bge-m3.


In [ ]:
# ============================================
# 11a) RAGAS testset generation
# ============================================
# Generates synthetic questions + reference answers from YOUR documents
# so you don't have to write them by hand.
#
# How it works:
#   1. Converts the `documents` list (already in memory) to LangChain format.
#   2. Builds a knowledge graph over the docs using the judge LLM + embeddings.
#   3. Synthesises diverse questions (simple, reasoning, multi-context).
#   4. Saves a JSON file in OUTPUT_DIR and wires it into CONFIG so the
#      eval cell below picks it up automatically.
#
# Cost estimate: ~$0.10-0.30 on gpt-4o-mini for 20 questions.
# Time estimate: 3-8 minutes depending on corpus size.
# Run this cell ONCE per corpus; reuse the saved JSON for every eval run.

TESTSET_SIZE = 20   # number of questions to generate
REGEN_TESTSET = False  # set True to overwrite an existing saved testset

from ragas.testset import TestsetGenerator
from langchain_core.documents import Document as LCDocument

testset_path = os.path.join(OUTPUT_DIR, f"ragas_testset_{RUN_ID}.json")

if not REGEN_TESTSET and CONFIG.get("evaluation_json_path") and os.path.exists(CONFIG["evaluation_json_path"]):
    print(f"Testset already exists at {CONFIG['evaluation_json_path']}")
    print("Set REGEN_TESTSET = True to regenerate.")
else:
    # Convert the notebook's `documents` list to LangChain Document objects.
    # `filename` in metadata is required by the RAGAS knowledge graph builder
    # to identify which chunks came from the same source document.
    lc_docs = []
    for doc in documents:
        text = (doc.get("text") or "").strip()
        if len(text) < 200:   # skip very short / empty documents
            continue
        lc_docs.append(LCDocument(
            page_content=text,
            metadata={
                "filename": doc.get("relative_path") or doc.get("doc_id", "unknown"),
                "doc_id": doc.get("doc_id", ""),
            }
        ))

    print(f"Using {len(lc_docs)} documents for testset generation "
          f"(skipped {len(documents) - len(lc_docs)} too-short docs)")
    if len(lc_docs) < 3:
        raise RuntimeError("Need at least 3 documents to generate a testset. "
                           "Check that your corpus loaded correctly.")

    # Build the generator (reuses judge_llm + judge_embeddings from cell 11b setup)
    tgen = TestsetGenerator(llm=judge_llm, embedding_model=judge_embeddings)

    print(f"Generating {TESTSET_SIZE} questions — this takes a few minutes...")
    testset = tgen.generate_with_langchain_docs(lc_docs, testset_size=TESTSET_SIZE)

    # Export to pandas so we can inspect and reformat
    df_testset = testset.to_pandas()
    print(f"\nGenerated {len(df_testset)} questions. Preview:")
    for _, row in df_testset.head(3).iterrows():
        print(f"  Q: {str(row.get('user_input', ''))[:100]}")
        print(f"  A: {str(row.get('reference', ''))[:100]}\n")

    # Convert to the eval JSON format expected by the RAGAS eval cell.
    # user_input  -> question
    # reference   -> gold_answer
    # (no gold_doc_ids from synthetic generation; leave as empty list)
    eval_items = []
    for _, row in df_testset.iterrows():
        q = str(row.get("user_input") or "").strip()
        a = str(row.get("reference") or "").strip()
        if q:
            eval_items.append({
                "question": q,
                "gold_answer": a if a else None,
                "gold_doc_ids": [],
            })

    with open(testset_path, "w", encoding="utf-8") as f:
        json.dump(eval_items, f, ensure_ascii=False, indent=2)
    print(f"Saved testset ({len(eval_items)} items): {testset_path}")

    # Wire it into CONFIG so the eval cell picks it up immediately.
    CONFIG["evaluation_json_path"] = testset_path
    print(f"CONFIG['evaluation_json_path'] set to: {testset_path}")

    # Also save the raw DataFrame for inspection / manual curation.
    raw_csv = testset_path.replace(".json", "_raw.csv")
    df_testset.to_csv(raw_csv, index=False, encoding="utf-8")
    print(f"Raw testset CSV (inspect + edit before reusing): {raw_csv}")


Using 12 documents for testset generation (skipped 0 too-short docs)
Generating 20 questions — this takes a few minutes...


Applying HeadlinesExtractor:   0%|          | 0/12 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/12 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/12 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/234 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/22 [00:00<?, ?it/s]


Generated 22 questions. Preview:
  Q: Who is Yun Xiong in the context of Retrieval-Augmented Generation?
  A: Yun Xiong is one of the authors of the paper titled 'Retrieval-Augmented Generation for Large Langua

  Q: What role does Dense Passage Retrieval play in Retrieval-Augmented Generation?
  A: Dense Passage Retrieval (DPR) is utilized on the retrieval side of Retrieval-Augmented Generation (R

  Q: How do Zhu et al. contribute to the advancements in retrieval systems for Large Language Models?
  A: Zhu et al. [2023] introduced the latest advancements in augmenting retrieval systems for Large Langu

Saved testset (22 items): /content/outputs/20260427_181401/ragas_testset_20260427_181401.json
CONFIG['evaluation_json_path'] set to: /content/outputs/20260427_181401/ragas_testset_20260427_181401.json
Raw testset CSV (inspect + edit before reusing): /content/outputs/20260427_181401/ragas_testset_20260427_181401_raw.csv


In [ ]:
# ============================================
# 11b) RAGAS evaluation - functions
# ============================================
# Pipeline:
#   1. For each question and each mode, call ask() once and cache the
#      generated answer + retrieved context texts. (This keeps the slow
#      Qwen-3B generation out of the RAGAS retry loop.)
#   2. Wrap every (q, answer, contexts, gold) tuple in a SingleTurnSample.
#   3. Build an EvaluationDataset and call evaluate().

from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithoutReference,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    FactualCorrectness,
)

def _extract_contexts(retrieved_fragments):
    """Return the list of context strings RAGAS expects."""
    out = []
    for r in retrieved_fragments or []:
        txt = (r.get("text") or "").strip()
        if txt:
            out.append(txt)
    return out

def generate_predictions(eval_items, mode):
    """Run ask() over the eval set; return a list of dicts ready for RAGAS.

    Each item: {question, answer, contexts, reference, confidence,
                low_confidence, retrieved_doc_ids, mode}.
    """
    rows = []
    for item in tqdm(eval_items, desc=f"Generating ({mode})"):
        q = item["question"]
        gold = item.get("gold_answer")
        try:
            r = ask(q, mode=mode, top_k=CONFIG["top_k_final"])
        except Exception as e:
            print(f"[warn] ask() failed on {q!r} ({mode}): {e!r}")
            r = {
                "answer": "", "retrieved_fragments": [],
                "confidence": None, "low_confidence": True,
            }
        rows.append({
            "question": q,
            "answer": r.get("answer", "") or "",
            "contexts": _extract_contexts(r.get("retrieved_fragments")),
            "reference": gold,
            "confidence": r.get("confidence"),
            "low_confidence": r.get("low_confidence"),
            "retrieved_doc_ids": [
                f.get("doc_id") for f in (r.get("retrieved_fragments") or [])
            ],
            "mode": mode,
        })
    return rows

def build_ragas_dataset(prediction_rows, has_reference):
    """Turn prediction rows into a RAGAS EvaluationDataset, skipping unusable rows."""
    samples, skipped = [], []
    for row in prediction_rows:
        if not row["contexts"] or not row["answer"]:
            skipped.append(row["question"])
            continue
        kwargs = dict(
            user_input=row["question"],
            response=row["answer"],
            retrieved_contexts=row["contexts"],
        )
        if has_reference and row.get("reference"):
            kwargs["reference"] = row["reference"]
        samples.append(SingleTurnSample(**kwargs))
    return EvaluationDataset(samples=samples), skipped

def run_ragas(prediction_rows):
    """Run RAGAS on a single mode's predictions; return (summary_dict, df)."""
    has_reference = any(r.get("reference") for r in prediction_rows)

    metrics = [
        Faithfulness(),
        ResponseRelevancy(),
        LLMContextPrecisionWithoutReference(),
    ]
    if has_reference:
        metrics += [
            LLMContextPrecisionWithReference(),
            LLMContextRecall(),
            FactualCorrectness(),
        ]

    dataset, skipped = build_ragas_dataset(prediction_rows, has_reference)
    if len(dataset.samples) == 0:
        return {"n_evaluated": 0, "skipped": skipped, "scores": {}}, pd.DataFrame()

    result = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=judge_llm,
        embeddings=judge_embeddings,
        show_progress=True,
    )
    df = result.to_pandas()

    # mean per metric column (skip non-numeric)
    score_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    summary = {
        "n_evaluated": int(len(df)),
        "n_skipped": len(skipped),
        "skipped_questions": skipped,
        "has_reference": bool(has_reference),
        "scores": {c: float(df[c].dropna().mean()) for c in score_cols if df[c].notna().any()},
    }
    return summary, df


In [ ]:
# ============================================
# 11c) RAGAS evaluation - run
# ============================================
# Runs the RAGAS evaluation on hierarchical retrieval mode.
# Saves:
#   * ragas_evaluation_report_<RUN_ID>.json  (summary + per-question scores)
#   * ragas_predictions_<RUN_ID>.csv         (raw per-question table)


eval_path = CONFIG.get("evaluation_json_path")
if eval_path and os.path.exists(eval_path):
    eval_items = json.load(open(eval_path, "r", encoding="utf-8"))
    print(f"Loaded {len(eval_items)} eval items from {eval_path}")

    has_any_gold = any(it.get("gold_answer") for it in eval_items)
    print(f"gold_answer present: {has_any_gold} → "
          f"{'full metric set' if has_any_gold else 'reference-free metrics only'}")

    # Step 1: generate answers using the local Qwen model
    print("\n=== Generating answers (hierarchical mode) ===")
    preds = generate_predictions(eval_items, mode="hierarchical")

    # Save raw predictions (re-run RAGAS scoring later without re-generating)
    preds_csv = os.path.join(OUTPUT_DIR, f"ragas_predictions_{RUN_ID}.csv")
    pd.DataFrame([{
        "question": p["question"],
        "answer": p["answer"],
        "n_contexts": len(p["contexts"]),
        "reference": p["reference"],
        "confidence": p["confidence"],
        "low_confidence": p["low_confidence"],
        "retrieved_doc_ids": ";".join([d for d in p["retrieved_doc_ids"] if d]),
    } for p in preds]).to_csv(preds_csv, index=False, encoding="utf-8")
    print(f"Saved predictions: {preds_csv}")

    # Step 2: score with RAGAS (judge = gpt-4o-mini)
    print("\n=== Scoring with RAGAS ===")
    summary, df = run_ragas(preds)

    # Collect per-row scores
    per_row = []
    if not df.empty:
        for _, row in df.iterrows():
            rec = {k: (float(v) if isinstance(v, (int, float, np.floating)) and not pd.isna(v) else None)
                   for k, v in row.items() if pd.api.types.is_numeric_dtype(type(v)) or isinstance(v, (int, float, np.floating))}
            rec["user_input"] = row.get("user_input")
            per_row.append(rec)

    # Print results
    print(f"\n{'='*56}")
    print("RAGAS results (hierarchical)")
    print(f"{'='*56}")
    for k, v in summary["scores"].items():
        print(f"  {k:42s}  {v:.4f}")
    print(f"  evaluated={summary['n_evaluated']}, skipped={summary['n_skipped']}")

    # Build and save report
    ragas_report = {
        "run_id": RUN_ID,
        "mode": "hierarchical",
        "judge": "gpt-4o-mini",
        "embedding_model": CONFIG["embedding_model_name"],
        "generator_model": CONFIG["generator_model_name"],
        "n_eval_items": len(eval_items),
        "summary": summary,
        "per_row": per_row,
    }

    ragas_path = os.path.join(OUTPUT_DIR, f"ragas_evaluation_report_{RUN_ID}.json")
    with open(ragas_path, "w", encoding="utf-8") as f:
        json.dump(ragas_report, f, ensure_ascii=False, indent=2, default=str)
    print(f"\nSaved RAGAS report: {ragas_path}")

else:
    print("No evaluation_json_path set in CONFIG (or file missing).\n"
          "Run cell 11a (testset generation) first, or set the path manually.")


Loaded 22 eval items from /content/outputs/20260427_181401/ragas_testset_20260427_181401.json
gold_answer present: True → full metric set

=== Generating answers (hierarchical mode) ===


Generating (hierarchical):   0%|          | 0/22 [00:00<?, ?it/s]

⚠️  Low confidence (score=0.22356167435646057). No relevant content found for this query.
Saved predictions: /content/outputs/20260427_181401/ragas_predictions_20260427_181401.csv

=== Scoring with RAGAS ===


Evaluating:   0%|          | 0/132 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[18]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-qS60zLxctVV2rPPVzJdKBqLu on tokens per min (TPM): Limit 200000, Used 195128, Requested 8293. Please try again in 1.026s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}})
ERROR:ragas.executor:Exception raised in Job[12]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-qS60zLxctVV2rPPVzJdKBqLu on tokens per min (TPM): Limit 200000, Used 197092, Requested 9388. Please try again in 1.944s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}})
ERROR:ragas.executor:Exception raised in Job[8]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[28]: RateLimitError(Error code: 429 - {'error': {'mes


RAGAS results (hierarchical)
  faithfulness                                0.8326
  answer_relevancy                            0.3661
  llm_context_precision_without_reference     0.8462
  llm_context_precision_with_reference        0.8969
  context_recall                              0.8509
  factual_correctness(mode=f1)                0.3518
  evaluated=22, skipped=0

Saved RAGAS report: /content/outputs/20260427_181401/ragas_evaluation_report_20260427_181401.json


In [ ]:

# ============================================
# 12) Example usage
# ============================================
query = "What does embedding do and which RAG models use it?"
result = ask(query, mode="hierarchical", top_k=3)

print("QUERY VIEWS:")
for qv in result["debug"]["query_views"]:
    print("-", qv)

#UPDATED : see scores
result = ask(query, mode="hierarchical", top_k=3)
print(f"Confidence score: {result['confidence']}")
print(f"Low confidence: {result['low_confidence']}")

print("\nANSWER:\n")
print(result["answer"])

print("\nSELECTED DOCUMENTS:")
for d in result["debug"]["selected_documents"][:10]:
    print(d["doc_id"], d["relative_path"], "score=", round(d["score"], 4))

print("\nCITATIONS:")
for c in result["citations"]:
    print(c)

single_query_json_path = os.path.join(CONFIG["output_dir"], f"single_query_result_{RUN_ID}.json")
json.dump(result, open(single_query_json_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("\nSaved single query JSON to:", single_query_json_path)


In [ ]:
from collections import OrderedDict, defaultdict
import textwrap
import re

def _is_prompt_leak(answer_text: str) -> bool:
    if not answer_text:
        return True
    markers = [
        "You are a careful research assistant",
        "Answer the question ONLY from the provided sources",
        "Every factual claim should be grounded",
        "Question:",
        "Sources:"
    ]
    return sum(m in answer_text for m in markers) >= 2

def _clean_text(text):
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    return text

def _shorten(text, width=260):
    return textwrap.shorten(_clean_text(text), width=width, placeholder=" ...")

def _build_source_map(result):
    source_map = {}

    for s in result.get("sources", []) or []:
        s_no = s.get("source_no") or s.get("id") or s.get("source")
        txt = s.get("text") or s.get("content") or s.get("snippet") or ""
        if s_no is not None:
            source_map[int(s_no)] = txt

    for s in result.get("retrieved_fragments", []) or []:
        s_no = s.get("source_no") or s.get("id")
        txt = s.get("text") or s.get("content") or s.get("snippet") or ""
        if s_no is not None and int(s_no) not in source_map:
            source_map[int(s_no)] = txt

    return source_map

def _infer_query_type(query: str) -> str:
    q = (query or "").strip().lower()
    if q.startswith("which document") or q.startswith("which documents"):
        return "document_discovery"
    if q.startswith("what is") or q.startswith("what are") or q.startswith("define") or q.startswith("explain"):
        return "definition"
    return "general_qa"

def _extract_definition_sentences(source_map, citations, max_sentences=6):
    """
    Pull likely definitional/explanatory sentences from retrieved text.
    Very lightweight heuristic fallback when LLM answer is missing.
    """
    candidate_sentences = []

    # prioritize first sources in citation order
    ordered_source_nos = []
    seen = set()
    for c in citations:
        s_no = c.get("source_no")
        if s_no is not None and s_no not in seen:
            ordered_source_nos.append(s_no)
            seen.add(s_no)

    definitional_markers = [
        "is formally termed",
        "is termed",
        "is called",
        "is defined as",
        "depends on",
        "it is assumed that",
        "not all positive instances have the same probability",
        "whether a positive instance is labeled depends",
        "labeling probability",
        "positive and unlabeled"
    ]

    for s_no in ordered_source_nos:
        txt = _clean_text(source_map.get(s_no, ""))
        if not txt:
            continue

        # crude sentence split
        sents = re.split(r'(?<=[\.\?\!])\s+', txt)
        for sent in sents:
            sent_clean = _clean_text(sent)
            if len(sent_clean) < 50:
                continue

            lower = sent_clean.lower()
            score = 0
            for m in definitional_markers:
                if m in lower:
                    score += 2
            if "instance-dependent" in lower:
                score += 2
            if "positive and unlabeled" in lower or "pu learning" in lower:
                score += 2
            if "abstract" in lower:
                score -= 1
            if "keywords" in lower:
                score -= 1

            if score > 0:
                candidate_sentences.append((score, s_no, sent_clean))

    # deduplicate by text
    dedup = []
    seen_text = set()
    for score, s_no, sent in sorted(candidate_sentences, key=lambda x: (-x[0], x[1])):
        key = sent.lower()
        if key not in seen_text:
            dedup.append((score, s_no, sent))
            seen_text.add(key)

    return dedup[:max_sentences]

def _fallback_synthesized_answer(query, result, source_map):
    qtype = _infer_query_type(query)
    citations = result.get("citations", []) or []
    selected_documents = result.get("selected_documents", []) or []

    # Build doc summary map
    by_doc = OrderedDict()
    for c in citations:
        doc_id = c.get("doc_id", "unknown_doc")
        if doc_id not in by_doc:
            by_doc[doc_id] = {
                "file": c.get("source_file_id") or c.get("relative_path") or "unknown_file",
                "pages": set(),
                "sources": []
            }
        by_doc[doc_id]["pages"].update(c.get("page_refs", []) or [])
        if c.get("source_no") is not None:
            by_doc[doc_id]["sources"].append(c["source_no"])

    if qtype == "document_discovery":
        lines = ["The following documents appear to discuss the requested topic:"]
        for i, (doc_id, meta) in enumerate(by_doc.items(), 1):
            pages = ", ".join(map(str, sorted(meta["pages"]))) if meta["pages"] else "n/a"
            srcs = ", ".join(f"[SOURCE {n}]" for n in sorted(set(meta["sources"])))
            lines.append(f"{i}. {meta['file']} ({doc_id}), pages {pages}. Evidence: {srcs}")
        return "\n".join(lines)

    # Definition / explanation fallback
    definitional = _extract_definition_sentences(source_map, citations, max_sentences=6)

    if qtype == "definition":
        if definitional:
            # pick strongest sentences and synthesize lightly
            parts = []
            used_sources = set()

            for _, s_no, sent in definitional[:4]:
                used_sources.add(s_no)
                parts.append(sent)

            # compress repeated wording
            merged = " ".join(parts)
            merged = re.sub(r'\s+', ' ', merged).strip()

            # very light cleanup of OCR-ish duplication
            merged = merged.replace("In instance-dependent PU learning, In instance-dependent PU learning,",
                                    "In instance-dependent PU learning,")
            merged = re.sub(r'(.{20,}?) \1', r'\1', merged)

            srcs = " ".join(f"[SOURCE {n}]" for n in sorted(used_sources))
            return f"{merged} {srcs}"

        # fallback if extraction failed
        if by_doc:
            doc_names = ", ".join(meta["file"] for _, meta in list(by_doc.items())[:3])
            return f"The retrieved sources suggest that the concept is discussed in documents such as {doc_names}, but a clean synthesized definition could not be extracted automatically from the retrieved passages."
        return "The evidence is insufficient to synthesize a grounded answer."

    # General QA fallback
    if definitional:
        used_sources = sorted({s_no for _, s_no, _ in definitional[:4]})
        text = " ".join(sent for _, _, sent in definitional[:4])
        srcs = " ".join(f"[SOURCE {n}]" for n in used_sources)
        return f"{text} {srcs}"

    if by_doc:
        lines = ["Relevant source documents were found, but a clean synthesized answer could not be generated automatically."]
        for doc_id, meta in list(by_doc.items())[:4]:
            srcs = ", ".join(f"[SOURCE {n}]" for n in sorted(set(meta["sources"])))
            lines.append(f"- {meta['file']} ({doc_id}) {srcs}")
        return "\n".join(lines)

    return "The evidence is insufficient to synthesize a grounded answer."

def render_human_answer_v2(result, max_docs=8, show_query_views=True, show_evidence=True):
    query = result.get("query", "(unknown query)")
    answer = result.get("answer", "") or ""
    query_views = result.get("query_views", []) or []
    selected_documents = result.get("selected_documents", []) or []
    citations = result.get("citations", []) or []

    # normalize selected docs
    norm_docs = []
    for d in selected_documents:
        if isinstance(d, dict):
            norm_docs.append(d)
        elif isinstance(d, (list, tuple)) and len(d) >= 2:
            norm_docs.append({"doc_id": d[0], "score": d[1]})
        else:
            norm_docs.append({"doc_id": str(d), "score": None})

    source_map = _build_source_map(result)

    by_doc = OrderedDict()
    for c in citations:
        doc_id = c.get("doc_id", "unknown_doc")
        if doc_id not in by_doc:
            by_doc[doc_id] = {
                "doc_id": doc_id,
                "file": c.get("source_file_id") or c.get("relative_path") or "unknown_file",
                "relative_path": c.get("relative_path") or c.get("source_file_id") or "unknown_file",
                "page_refs": set(),
                "source_nos": [],
                "anchor_chunk_ids": [],
            }
        by_doc[doc_id]["page_refs"].update(c.get("page_refs", []) or [])
        if c.get("source_no") is not None:
            by_doc[doc_id]["source_nos"].append(c["source_no"])
        if c.get("anchor_chunk_id"):
            by_doc[doc_id]["anchor_chunk_ids"].append(c["anchor_chunk_id"])

    # merge doc scores
    score_map = {}
    for d in norm_docs:
        if d.get("doc_id") is not None:
            score_map[d["doc_id"]] = d.get("score")

    doc_rows = []
    for doc_id, meta in by_doc.items():
        doc_rows.append({
            "doc_id": doc_id,
            "file": meta["file"],
            "relative_path": meta["relative_path"],
            "score": score_map.get(doc_id),
            "page_refs": sorted(meta["page_refs"]),
            "source_nos": sorted(set(meta["source_nos"])),
            "anchor_chunk_ids": list(OrderedDict.fromkeys(meta["anchor_chunk_ids"])),
        })

    def sort_key(x):
        score = x["score"]
        return (-(score if isinstance(score, (int, float)) else -1), x["file"])

    doc_rows = sorted(doc_rows, key=sort_key)[:max_docs]

    print("=" * 100)
    print("FINAL HUMAN-READABLE RESPONSE")
    print("=" * 100)
    print(f"\nQuery:\n{query}\n")

    if show_query_views and query_views:
        print("Interpreted query views:")
        for qv in query_views:
            print(f"  - {qv}")
        print()

    print("Synthesized answer:")
    if answer and not _is_prompt_leak(answer):
        print(textwrap.fill(_clean_text(answer), width=100))
    else:
        fallback = _fallback_synthesized_answer(query, result, source_map)
        print(textwrap.fill(_clean_text(fallback), width=100))
    print()

    print("Structured source summary:")
    if doc_rows:
        for i, row in enumerate(doc_rows, 1):
            score_txt = f"{row['score']:.4f}" if isinstance(row["score"], (int, float)) else "n/a"
            pages_txt = ", ".join(map(str, row["page_refs"])) if row["page_refs"] else "n/a"
            source_refs = ", ".join(f"[SOURCE {n}]" for n in row["source_nos"]) if row["source_nos"] else "n/a"
            print(f"  {i}. File: {row['file']}")
            print(f"     doc_id: {row['doc_id']}")
            print(f"     score: {score_txt}")
            print(f"     pages: {pages_txt}")
            print(f"     citations: {source_refs}")
            print(f"     anchor chunks: {', '.join(row['anchor_chunk_ids'][:5]) if row['anchor_chunk_ids'] else 'n/a'}")
            print()
    else:
        print("  No source documents selected.\n")

    if show_evidence and citations:
        print("Evidence details:")
        shown = set()
        for c in citations:
            source_no = c.get("source_no", "?")
            if source_no in shown:
                continue
            shown.add(source_no)
            file_name = c.get("source_file_id") or c.get("relative_path") or "unknown_file"
            doc_id = c.get("doc_id", "unknown_doc")
            pages = c.get("page_refs", []) or []
            anchor = c.get("anchor_chunk_id", "n/a")
            snippet = _shorten(source_map.get(source_no, ""), 260)
            print(f"  [SOURCE {source_no}] {file_name} | {doc_id} | pages={pages} | anchor={anchor}")
            if snippet:
                print(f"    snippet: {snippet}")
        print()

    print("=" * 100)

# Run it
render_human_answer_v2(result)

In [ ]:
#UPDATED
# ============================================
# 13) Download outputs and cache as ZIP folders
# ============================================
import os
import shutil
from google.colab import files

print("Zipping folders...")

# 1. Zip the outputs folder
# This will create a file named something like 'outputs_20241025_153000.zip'
output_zip_path = f"/content/outputs_{RUN_ID}"
shutil.make_archive(output_zip_path, 'zip', CONFIG["output_dir"])

# 2. Zip the cache folder (contains faiss.index, embeddings.npy, chunks.json)
cache_zip_path = f"/content/cache_{RUN_ID}"
shutil.make_archive(cache_zip_path, 'zip', CACHE_DIR)

# 3. Download the zipped folders
print(f"Downloading outputs archive: {output_zip_path}.zip")
files.download(f"{output_zip_path}.zip")

print(f"Downloading cache archive: {cache_zip_path}.zip")
files.download(f"{cache_zip_path}.zip")

Zipping folders...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>